In [56]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import random
import time
import asyncio

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import OrderedDict, deque

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("Environment initialized successfully.")

Environment initialized successfully.


In [57]:
# ============================================================
# CELL 2 — DNS CACHE
# ============================================================

class DNSCache:

    def __init__(self, capacity=20):
        self.capacity = capacity
        self.entries = OrderedDict()

    def lookup(self, domain, current_time):

        # Domain not present
        if domain not in self.entries:
            return False, "MISS"

        entry = self.entries[domain]

        # TTL expired
        if current_time >= entry["expires_at"]:
            del self.entries[domain]
            return False, "EXPIRED"

        # LRU update
        self.entries.move_to_end(domain)

        return True, "HIT"

    def insert(self, domain, ttl, current_time):

        # Remove existing entry
        if domain in self.entries:
            del self.entries[domain]

        # LRU eviction
        while len(self.entries) >= self.capacity:
            self.entries.popitem(last=False)

        self.entries[domain] = {
            "created_at": current_time,
            "expires_at": current_time + ttl
        }

    def size(self):
        return len(self.entries)

    def clear(self):
        self.entries.clear()

    def contents(self):
        return list(self.entries.keys())

In [58]:
# ============================================================
# CELL 3 — DNS NETWORK SIMULATOR
# ============================================================

class DNSNetworkSimulator:

    def __init__(self):

        self.domains = [
            "google.com",
            "github.com",
            "youtube.com",
            "amazon.com",
            "wikipedia.org",
            "microsoft.com",
            "apple.com",
            "linkedin.com",
            "reddit.com",
            "stackoverflow.com",
            "cloudflare.com",
            "python.org",
            "openai.com",
            "netflix.com",
            "mozilla.org",
            "ubuntu.com",
            "docker.com",
            "ibm.com",
            "aws.amazon.com",
            "facebook.com"
        ]

        self.reset()

    # --------------------------------------------------------
    # RESET
    # --------------------------------------------------------

    def reset(self):

        self.cache = DNSCache(
            capacity=20
        )

        self.simulation_time = 0
        self.request_id = 0

        self.previous_domain = None

        self.total_requests = 0
        self.cache_hits = 0
        self.cache_misses = 0
        self.expired_entries = 0

        self.upstream_queries = 0
        self.packet_losses = 0
        self.retransmissions = 0

        self.latencies = deque(
            maxlen=200
        )

        self.history = deque(
            maxlen=200
        )

    # --------------------------------------------------------
    # DOMAIN SELECTION
    # --------------------------------------------------------

    def choose_domain(self, locality):

        # Higher locality means users repeatedly
        # access the same domain.

        if (
            self.previous_domain is not None
            and random.random() < locality
        ):
            domain = self.previous_domain

        else:
            domain = random.choice(
                self.domains
            )

        self.previous_domain = domain

        return domain

    # --------------------------------------------------------
    # PROCESS ONE REQUEST
    # --------------------------------------------------------

    def process_request(
        self,
        cache_size,
        ttl,
        locality,
        network_latency,
        jitter,
        packet_loss
    ):

        # Apply changed cache capacity immediately
        self.cache.capacity = cache_size

        self.request_id += 1
        self.total_requests += 1

        domain = self.choose_domain(
            locality
        )

        # ----------------------------------------------------
        # CACHE LOOKUP
        # ----------------------------------------------------

        hit, lookup_status = self.cache.lookup(
            domain,
            self.simulation_time
        )

        if hit:

            self.cache_hits += 1

            # Cache response is very fast
            latency = 0.2

            event = "CACHE HIT"

        else:

            self.cache_misses += 1
            self.upstream_queries += 1

            # ------------------------------------------------
            # NETWORK DELAY
            # ------------------------------------------------

            delay = max(
                0,
                network_latency +
                random.uniform(
                    -jitter,
                    jitter
                )
            )

            # ------------------------------------------------
            # PACKET LOSS
            # ------------------------------------------------

            lost = (
                random.random()
                < packet_loss
            )

            if lost:

                self.packet_losses += 1
                self.retransmissions += 1

                # Retransmission delay
                retry_delay = max(
                    0,
                    network_latency +
                    random.uniform(
                        -jitter,
                        jitter
                    )
                )

                latency = (
                    delay +
                    retry_delay
                )

                event = "RETRANSMISSION"

            else:

                latency = delay

                if lookup_status == "EXPIRED":
                    self.expired_entries += 1
                    event = "TTL EXPIRED"

                else:
                    event = "DNS QUERY"

            # ------------------------------------------------
            # STORE DNS RESULT
            # ------------------------------------------------

            self.cache.insert(
                domain,
                ttl,
                self.simulation_time
            )

        # ----------------------------------------------------
        # STORE METRICS
        # ----------------------------------------------------

        self.latencies.append(
            latency
        )

        self.history.append({

            "request": self.request_id,

            "time": self.simulation_time,

            "domain": domain,

            "event": event,

            "latency": latency,

            "cache_entries":
                self.cache.size()
        })

        self.simulation_time += 1

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    def get_metrics(self):

        total = self.total_requests

        if total == 0:

            return {
                "requests": 0,
                "hit_ratio": 0,
                "miss_ratio": 0,
                "avg_latency": 0,
                "p95_latency": 0,
                "upstream": 0,
                "losses": 0,
                "retransmissions": 0,
                "expired": 0,
                "cache_size": self.cache.size()
            }

        latency_values = list(
            self.latencies
        )

        return {

            "requests":
                total,

            "hit_ratio":
                self.cache_hits / total,

            "miss_ratio":
                self.cache_misses / total,

            "avg_latency":
                np.mean(
                    latency_values
                ),

            "p95_latency":
                np.percentile(
                    latency_values,
                    95
                ),

            "upstream":
                self.upstream_queries,

            "losses":
                self.packet_losses,

            "retransmissions":
                self.retransmissions,

            "expired":
                self.expired_entries,

            "cache_size":
                self.cache.size()
        }


sim = DNSNetworkSimulator()

print("DNS network simulator created.")

DNS network simulator created.


In [59]:
# ============================================================
# CELL 4 — NETWORK PARAMETERS
# ============================================================

cache_size = widgets.IntSlider(
    value=20,
    min=2,
    max=100,
    step=1,
    description="Cache Size",
    continuous_update=True,
    layout=widgets.Layout(width="500px")
)

ttl = widgets.IntSlider(
    value=10,
    min=1,
    max=60,
    step=1,
    description="TTL (s)",
    continuous_update=True,
    layout=widgets.Layout(width="500px")
)

locality = widgets.FloatSlider(
    value=0.70,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Locality",
    readout_format=".0%",
    continuous_update=True,
    layout=widgets.Layout(width="500px")
)

network_latency = widgets.IntSlider(
    value=30,
    min=1,
    max=200,
    step=5,
    description="Latency (ms)",
    continuous_update=True,
    layout=widgets.Layout(width="500px")
)

jitter = widgets.IntSlider(
    value=5,
    min=0,
    max=50,
    step=1,
    description="Jitter (ms)",
    continuous_update=True,
    layout=widgets.Layout(width="500px")
)

packet_loss = widgets.FloatSlider(
    value=0.02,
    min=0.0,
    max=0.30,
    step=0.01,
    description="Packet Loss",
    readout_format=".0%",
    continuous_update=True,
    layout=widgets.Layout(width="500px")
)

requests_per_step = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="Requests/Step",
    layout=widgets.Layout(width="500px")
)

simulation_speed = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=2.0,
    step=0.1,
    description="Update (s)",
    layout=widgets.Layout(width="500px")
)

print("Parameters created.")

Parameters created.


In [60]:
# ============================================================
# CELL 5 — DASHBOARD OUTPUT
# ============================================================

dashboard_output = widgets.Output(
    layout=widgets.Layout(
        width="100%"
    )
)

status_output = widgets.HTML(
    value="<b>Status:</b> Ready"
)

print("Dashboard output initialized.")

Dashboard output initialized.


In [61]:
# ============================================================
# CELL 6 — DASHBOARD RENDERER
# ============================================================

def render_dashboard():

    with dashboard_output:

        clear_output(wait=True)

        metrics = sim.get_metrics()

        history = list(
            sim.history
        )

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(15, 9)
        )

        # ====================================================
        # GRAPH 1 — LATENCY
        # ====================================================

        if history:

            request_numbers = [
                x["request"]
                for x in history
            ]

            latency_values = [
                x["latency"]
                for x in history
            ]

            axes[0, 0].plot(
                request_numbers,
                latency_values,
                linewidth=1.8
            )

        axes[0, 0].set_title(
            "Network Response Latency"
        )

        axes[0, 0].set_xlabel(
            "Request"
        )

        axes[0, 0].set_ylabel(
            "Latency (ms)"
        )

        axes[0, 0].grid(
            alpha=0.3
        )

        # ====================================================
        # GRAPH 2 — CACHE PERFORMANCE
        # ====================================================

        axes[0, 1].bar(
            ["Cache HIT", "Cache MISS"],
            [
                sim.cache_hits,
                sim.cache_misses
            ]
        )

        axes[0, 1].set_title(
            f"Cache Performance "
            f"({metrics['hit_ratio']:.1%} Hit Ratio)"
        )

        axes[0, 1].set_ylabel(
            "Requests"
        )

        # ====================================================
        # GRAPH 3 — NETWORK TRAFFIC
        # ====================================================

        if history:

            cumulative_upstream = np.cumsum([

                0
                if x["event"] == "CACHE HIT"
                else 1

                for x in history

            ])

            axes[1, 0].plot(
                request_numbers,
                cumulative_upstream,
                linewidth=2
            )

        axes[1, 0].set_title(
            "Upstream DNS Traffic"
        )

        axes[1, 0].set_xlabel(
            "Request"
        )

        axes[1, 0].set_ylabel(
            "DNS Queries"
        )

        axes[1, 0].grid(
            alpha=0.3
        )

        # ====================================================
        # GRAPH 4 — NETWORK EVENTS
        # ====================================================

        event_counts = {

            "CACHE HIT":
                0,

            "DNS QUERY":
                0,

            "TTL EXPIRED":
                0,

            "RETRANSMISSION":
                0
        }

        for item in history:

            event = item["event"]

            if event in event_counts:
                event_counts[event] += 1

        axes[1, 1].bar(
            list(event_counts.keys()),
            list(event_counts.values())
        )

        axes[1, 1].set_title(
            "Network Events"
        )

        axes[1, 1].set_ylabel(
            "Count"
        )

        axes[1, 1].tick_params(
            axis="x",
            rotation=25
        )

        # ====================================================
        # LAYOUT
        # ====================================================

        plt.tight_layout()

        plt.show()

        # ====================================================
        # METRIC CARDS
        # ====================================================

        print(
            f"""
╔══════════════════════════════════════════════════════════╗
║                 DNS NETWORK SIMULATOR                    ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Requests              : {metrics['requests']:<28}║
║  Cache Hit Ratio       : {metrics['hit_ratio']:<27.2%}║
║  Average Latency       : {metrics['avg_latency']:<24.2f} ms ║
║  P95 Latency           : {metrics['p95_latency']:<24.2f} ms ║
║  Upstream DNS Queries  : {metrics['upstream']:<28}║
║  Packet Loss Events    : {metrics['losses']:<28}║
║  Retransmissions       : {metrics['retransmissions']:<28}║
║  TTL Expirations       : {metrics['expired']:<28}║
║  Current Cache Entries : {metrics['cache_size']:<28}║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
"""
        )

        # ====================================================
        # CURRENT CONFIGURATION
        # ====================================================

        print(
            f"""
Current Network Configuration
──────────────────────────────────────────────────────────

Cache Capacity    : {cache_size.value}
TTL               : {ttl.value} s
Query Locality    : {locality.value:.0%}
Network Latency   : {network_latency.value} ms
Jitter            : {jitter.value} ms
Packet Loss       : {packet_loss.value:.0%}
"""
        )

        # ====================================================
        # RECENT REQUESTS
        # ====================================================

        if history:

            recent = pd.DataFrame(
                history[-10:]
            )

            print(
                "\nRecent Network Requests"
            )

            display(
                recent[
                    [
                        "request",
                        "domain",
                        "event",
                        "latency",
                        "cache_entries"
                    ]
                ].tail(10)
            )

In [62]:
# ============================================================
# CELL 7 — SIMULATION STEP
# ============================================================

def simulation_step():

    for _ in range(
        requests_per_step.value
    ):

        sim.process_request(

            cache_size=
                cache_size.value,

            ttl=
                ttl.value,

            locality=
                locality.value,

            network_latency=
                network_latency.value,

            jitter=
                jitter.value,

            packet_loss=
                packet_loss.value
        )

    render_dashboard()

In [63]:
# ============================================================
# CELL 8 — CONTROL BUTTONS
# ============================================================

step_button = widgets.Button(
    description="STEP",
    button_style="primary",
    icon="forward"
)

start_button = widgets.Button(
    description="START LIVE",
    button_style="success",
    icon="play"
)

stop_button = widgets.Button(
    description="STOP",
    button_style="danger",
    icon="stop"
)

reset_button = widgets.Button(
    description="RESET",
    button_style="warning",
    icon="refresh"
)

print("Controls created.")

Controls created.


In [64]:
# ============================================================
# CELL 9 — BUTTON CALLBACKS
# ============================================================

running = False


def step_clicked(button):

    simulation_step()


def reset_clicked(button):

    global running

    running = False

    sim.reset()

    status_output.value = (
        "<b>Status:</b> "
        "<span style='color:#d97706;'>Reset</span>"
    )

    render_dashboard()


async def live_loop():

    global running

    while running:

        simulation_step()

        await asyncio.sleep(
            simulation_speed.value
        )


def start_clicked(button):

    global running

    if running:
        return

    running = True

    status_output.value = (
        "<b>Status:</b> "
        "<span style='color:green;'>LIVE SIMULATION RUNNING</span>"
    )

    asyncio.create_task(
        live_loop()
    )


def stop_clicked(button):

    global running

    running = False

    status_output.value = (
        "<b>Status:</b> "
        "<span style='color:red;'>Simulation stopped</span>"
    )


step_button.on_click(
    step_clicked
)

start_button.on_click(
    start_clicked
)

stop_button.on_click(
    stop_clicked
)

reset_button.on_click(
    reset_clicked
)

print("Callbacks connected.")

Callbacks connected.


In [65]:
# ============================================================
# CELL 10 — DYNAMIC PARAMETER UPDATES
# ============================================================

def parameter_changed(change):

    # If simulation has not started yet,
    # still show the current configuration.

    render_dashboard()


for control in [

    cache_size,
    ttl,
    locality,
    network_latency,
    jitter,
    packet_loss,
    requests_per_step,
    simulation_speed

]:

    control.observe(
        parameter_changed,
        names="value"
    )

print(
    "Dynamic parameter callbacks connected."
)

Dynamic parameter callbacks connected.


In [66]:
# ============================================================
# CELL 11 — DASHBOARD UI
# ============================================================

title = widgets.HTML(
    """
    <h1 style="margin-bottom:5px;">
        DNS Network Simulator
    </h1>

    <p style="margin-top:0;">
        Interactive simulation of DNS caching,
        latency, TTL, locality and packet loss
    </p>
    """
)


parameter_panel = widgets.VBox([

    widgets.HTML(
        "<h3>Network Parameters</h3>"
    ),

    cache_size,
    ttl,
    locality,
    network_latency,
    jitter,
    packet_loss,

    widgets.HTML(
        "<h3>Simulation Controls</h3>"
    ),

    requests_per_step,
    simulation_speed,

    widgets.HBox([
        step_button,
        start_button,
        stop_button,
        reset_button
    ]),

    status_output
])


dashboard = widgets.VBox([

    title,

    parameter_panel,

    widgets.HTML(
        "<hr><h3>Live Network Dashboard</h3>"
    ),

    dashboard_output
])


display(dashboard)

In [67]:
# ============================================================
# CELL 12 — INITIALIZE
# ============================================================

render_dashboard()

print(
    "\nDashboard ready."
)

print(
    "Click STEP to generate requests."
)

print(
    "Click START LIVE for continuous simulation."
)


Dashboard ready.
Click STEP to generate requests.
Click START LIVE for continuous simulation.
